In [15]:
import pandas as pd
import numpy as np
from linearmodels.iv import IV2SLS

In [16]:
def flip_dataframe(file_path):
    # Read the CSV without interpreting the first row as headers.
    df = pd.read_csv(file_path, header=None)
    # Transpose the DataFrame.
    df_transposed = df.T
    # Set the first row of the transposed DataFrame as header.
    df_transposed.columns = df_transposed.iloc[0]
    # Drop the first row (now redundant as header) and reset the index.
    df_transposed = df_transposed.drop(df_transposed.index[0]).reset_index(drop=True)
    return df_transposed

# Example usage for one of your files:
dbs = flip_dataframe("../data/dbs ratios.csv")
ocbc = flip_dataframe("../data/ocbc ratios.csv")
uob = flip_dataframe("../data/uob ratios.csv")

In [ ]:
#dbs.to_csv('../data/dbs_clean.csv', index=False)
#ocbc.to_csv('../data/ocbc_clean.csv', index=False)
#uob.to_csv('../data/uob_clean.csv', index=False)

In [18]:
dbs['bank'] = 'DBS'
ocbc['bank'] = 'OCBC'
uob['bank'] = 'UOB'
# Concatenate the three banks into one DataFrame
df_banks = pd.concat([dbs, ocbc, uob], ignore_index=True)

df_banks.rename(columns={df_banks.columns[0]: 'Year'}, inplace=True)
df_banks['Year'] = pd.to_datetime(df_banks['Year'], format='%d/%m/%Y', errors='coerce').dt.year.astype(int)

df_banks = df_banks.apply(lambda col: col if col.name == 'bank' 
                          else pd.to_numeric(col.astype(str).str.replace(',', ''), errors='coerce'))


In [19]:
print(df_banks.columns.tolist())

['Year', ' ∟ Total assets (thousands USD)', ' ∟ Total equity / Total assets (%)', ' ∟ Total operating expenses (thousands USD)', ' ∟ Net interest margin (average total earning assets) (%)', ' ∟ Net interest income (expenses) / Average total assets (%)', ' ∟ Non-interest income / Average total assets (%)', ' ∟ Non-interest expenses / Average total assets (%)', ' ∟ Pre-tax operating profit / Average total assets (%)', ' ∟ Non-operating income (expense) & taxes / Average total assets (%)', ' ∟ Return on average assets (ROAA) (%)', ' ∟ Return on average equity (ROAE) (%)', ' ∟ Dividends related to the period / Net income (Dividend pay-out ratio) (%)', ' ∟ Net profit post dividends / Average Equity (%)', ' ∟ Other non-operating income (expense) / Net income (%)', ' ∟ Cost-to-income (Efficiency) ratio (%)', ' ∟ Recurring earning power (Pre-impairment operating profit / Average total assets) (%)', 'bank', ' ∟ Total assets', ' ∟ Total operating expenses']


In [20]:
# Create a mask for UOB rows
mask_uob = df_banks['bank'] == 'UOB'

# --- For Total Assets ---
# If your desired column is already named 'total assets (thousands USD)' and UOB's asset data is in 'Total Assets'
if ' ∟ Total assets' in df_banks.columns:
    # For UOB rows, fill missing values in the main asset column using the extra column
    if ' ∟ Total assets (thousands USD)' in df_banks.columns:
        df_banks.loc[mask_uob, ' ∟ Total assets (thousands USD)'] = df_banks.loc[mask_uob, ' ∟ Total assets (thousands USD)'].fillna(
            df_banks.loc[mask_uob, ' ∟ Total assets']
        )
    else:
        # If the desired column doesn't exist for UOB, just copy the values
        df_banks.loc[mask_uob, ' ∟ Total assets (thousands USD)'] = df_banks.loc[mask_uob, ' ∟ Total assets']
    # Drop the extra column
    df_banks.drop(columns=[' ∟ Total assets'], inplace=True)

# --- For Total Operating Expenses ---
# Assume the desired column is named 'total operating expenses (thousands USD)' and UOB's data is stored in an extra column, say 'Total Operating Expenses'
if ' ∟ Total operating expenses' in df_banks.columns:
    if ' ∟ Total operating expenses (thousands USD)' in df_banks.columns:
        df_banks.loc[mask_uob, ' ∟ Total operating expenses (thousands USD)'] = df_banks.loc[mask_uob, ' ∟ Total operating expenses (thousands USD)'].fillna(
            df_banks.loc[mask_uob, ' ∟ Total operating expenses']
        )
    else:
        df_banks.loc[mask_uob, ' ∟ Total operating expenses (thousands USD)'] = df_banks.loc[mask_uob, ' ∟ Total operating expenses']
    df_banks.drop(columns=[' ∟ Total operating expenses'], inplace=True)

# Verify the change
df_banks

,Year,∟ Total assets (thousands USD),∟ Total equity / Total assets (%),∟ Total operating expenses (thousands USD),∟ Net interest margin (average total earning assets) (%),∟ Net interest income (expenses) / Average total assets (%),∟ Non-interest income / Average total assets (%),∟ Non-interest expenses / Average total assets (%),∟ Pre-tax operating profit / Average total assets (%),∟ Non-operating income (expense) & taxes / Average total assets (%),∟ Return on average assets (ROAA) (%),∟ Return on average equity (ROAE) (%),∟ Dividends related to the period / Net income (Dividend pay-out ratio) (%),∟ Net profit post dividends / Average Equity (%),∟ Other non-operating income (expense) / Net income (%),∟ Cost-to-income (Efficiency) ratio (%),∟ Recurring earning power (Pre-impairment operating profit / Average total assets) (%),bank
0,2024,608115141.0,8.32,6646328.0,2.07,1.84,1.01,1.23,1.64,-0.17,1.44,17.23,55.87,7.60,2.24,40.55,1.69,DBS
1,2023,560671177.0,8.42,6294555.0,2.04,1.84,0.88,1.18,1.55,-0.18,1.36,16.87,46.11,9.09,0.89,41.17,1.60,DBS
2,2022,552854366.0,7.68,5272943.0,1.71,1.53,0.78,1.04,1.31,-0.13,1.15,14.28,45.20,7.82,3.58,42.96,1.32,DBS
3,2021,507563087.0,8.41,4859806.0,1.42,1.26,0.86,0.99,1.16,-0.12,1.02,12.12,38.39,7.47,2.72,46.30,1.14,DBS
4,2020,491595199.0,8.41,4657742.0,1.63,1.48,0.90,1.47,0.87,-0.13,0.77,8.94,54.46,4.07,-3.76,42.20,1.37,DBS
5,2019,429740213.0,8.95,4645190.0,1.84,1.70,0.87,1.23,1.34,-0.21,1.14,12.65,59.65,5.10,-0.08,43.03,1.47,DBS
6,2018,403539717.0,9.06,4259965.0,1.82,1.68,0.79,1.21,1.25,-0.20,1.06,11.34,54.41,5.17,-0.94,44.10,1.38,DBS
7,2017,387334275.0,9.62,3894209.0,1.70,1.56,0.90,1.39,1.04,-0.17,0.90,9.31,16.94,7.73,-3.95,42.41,1.41,DBS
8,2016,332966880.0,9.75,3437738.0,1.69,1.56,0.89,1.27,1.08,-0.25,0.93,9.71,17.22,8.04,-9.95,43.28,1.39,DBS
9,2015,323809318.0,9.35,3465592.0,1.72,1.58,0.85,1.25,1.18,-0.16,1.02,11.00,16.49,9.19,0.20,44.86,1.34,DBS


### digital payment

In [21]:
xls = pd.ExcelFile("../data/Retail_Payment_Analysis.xlsx")
raw_data = xls.parse('Raw Data')

atm as BNDP and rest as TTP

In [22]:
raw_data['Category'] = raw_data['Instrument'].apply(lambda x: 'BNDP' 
                                                    if 
                                                        'ATM Withdrawals' in x or 
                                                        'CNP Card Payments (Credit)' in x or
                                                        'CNP Card Payments (Debit)' in x or
                                                        'FAST' in x or 
                                                        'Inter-bank GIRO' in x or 
                                                        'POS Card Payments (Credit)' in x or 
                                                        'POS Card Payments (Debit)' in x or 
                                                        'Singapore Dollar Cheque Clearing' in x or 
                                                        'Stored Value Facilities' in x or 
                                                        'US Dollar Cheque Clearing' in x 
                                                    else 'TPP')
print("Unique categories in raw_data:", raw_data['Instrument'].unique())

Unique categories in raw_data: ['ATM Withdrawals' 'CNP Card Payments (Credit)'
 'CNP Card Payments (Debit)' 'FAST' 'Inter-bank GIRO'
 'POS Card Payments (Credit)' 'POS Card Payments (Debit)'
 'Singapore Dollar Cheque Clearing' 'Stored Value Facilities'
 'US Dollar Cheque Clearing']


In [23]:

agg = raw_data.groupby(['Year', 'Category'])['Value'].sum().unstack()

# Rename columns for clarity and compute logs:
agg = agg.rename(columns={'BNDP': 'bank_involved', 'TPP': 'third_party'})
agg['LNBDP'] = np.log(agg['bank_involved'])
#agg['LNTPP'] = np.log(agg['third_party'])
agg = agg.reset_index()[['Year', 'LNBDP']]
agg['Year'] = agg['Year'].astype(int)
agg

Category,Year,LNBDP
0,2014,14.019855
1,2015,14.011957
2,2016,14.017553
3,2017,14.001045
4,2018,14.013493
5,2019,14.099713
6,2020,13.936822
7,2021,13.930941
8,2022,13.934624
9,2023,13.939717


# Combining dataset

In [24]:
df_banks['SIZE'] = np.log(df_banks[' ∟ Total assets (thousands USD)'])
df_banks['RISK'] = df_banks[' ∟ Total equity / Total assets (%)'] / df_banks[' ∟ Total assets (thousands USD)']
df_banks['COST'] = df_banks[' ∟ Total operating expenses (thousands USD)'] / df_banks[' ∟ Total assets (thousands USD)']

In [25]:
print("df_banks Year values and type:")
print(df_banks['Year'].head())
print(df_banks['Year'].dtype)

print("\nagg Year values and type:")
print(agg['Year'].head())
print(agg['Year'].dtype)

df_banks Year values and type:
0    2024
1    2023
2    2022
3    2021
4    2020
Name: Year, dtype: int64
int64

agg Year values and type:
0    2014
1    2015
2    2016
3    2017
4    2018
Name: Year, dtype: int64
int64


merge bank with digital payment

In [26]:
# Merge on 'Year'
df_panel = pd.merge(df_banks, agg, on='Year', how='left')

# Set a multi-index with bank and year for panel data operations
df_panel = df_panel.set_index(['bank', 'Year']).sort_index()
df_panel

∟ Total assets (thousands USD)  \
bank Year                                    
DBS  2005                      108282660.0   
     2006                      128664923.0   
     2007                      161645150.0   
     2008                      178375490.0   
     2009                      184298135.0   
     2010                      220271742.0   
     2011                      262048889.0   
     2012                      288543516.0   
     2013                      317717542.0   
     2014                      333509423.0   
     2015                      323809318.0   
     2016                      332966880.0   
     2017                      387334275.0   
     2018                      403539717.0   
     2019                      429740213.0   
     2020                      491595199.0   
     2021                      507563087.0   
     2022                      552854366.0   
     2023                      560671177.0   
     2024                      608115141.0   
OCBC 2005                       80946049.0   
     2006                       98578660.0   
     2007                      121154097.0   
     2008                      126031932.0   
     2009                      138449153.0   
     2010                      178014606.0   
     2011                      213544708.0   
     2012                      241882674.0   
     2013                      267484699.0   
     2014                      303660048.0   
     2015                      275966926.0   
     2016                      283401479.0   
     2017                      340369841.0   
     2018                      342572534.0   
     2019                      364972194.0   
     2020                      394368814.0   
     2021                      401114907.0   
     2022                      414193058.0   
     2023                      440940400.0   
UOB  2005                       87172679.0   
     2006                      105157543.0   
     2007                      121392092.0   
     2008                      127112868.0   
     2009                      132234843.0   
     2010                      165977083.0   
     2011                      182177088.0   
     2012                      206701681.0   
     2013                      224633742.0   
     2014                      232147236.0   
     2015                      223503219.0   
     2016                      235101730.0   
     2017                      268286965.0   
     2018                      284358411.0   
     2019                      300184524.0   
     2020                      326612214.0   
     2021                      339811361.0   
     2022                      375026021.0   
     2023                      397027158.0   

            ∟ Total equity / Total assets (%)  \
bank Year                                       
DBS  2005                               10.65   
     2006                               10.66   
     2007                                9.94   
     2008                                9.35   
     2009                               11.41   
     2010                               11.67   
     2011                                9.70   
     2012                               10.20   
     2013                                9.37   
     2014                                9.12   
     2015                                9.35   
     2016                                9.75   
     2017                                9.62   
     2018                                9.06   
     2019                                8.95   
     2020                                8.41   
     2021                                8.41   
     2022                                7.68   
     2023                                8.42   
     2024                                8.32   
OCBC 2005                               10.01   
     2006                                9.58   
     2007                                9.64   
     2008                 

generate lags and first differences for DIFF GMM estimation

In [27]:
# Here we demonstrate for ROA as the dependent variable.
df_panel['ROA'] = df_panel[" ∟ Return on average assets (ROAA) (%)"]
# Generate lagged ROA variables (lag 1 and lag 2)
df_panel['ROA_lag1'] = df_panel.groupby(level='bank')['ROA'].shift(1)
df_panel['ROA_lag2'] = df_panel.groupby(level='bank')['ROA'].shift(2)

# Compute first differences
df_panel['d_ROA'] = df_panel['ROA'] - df_panel['ROA_lag1']
# The lagged difference (for the endogenous regressor) is:
df_panel['d_ROA_lag1'] = df_panel['ROA_lag1'] - df_panel.groupby(level='bank')['ROA'].shift(2)

# Similarly, create differences for the control variables
for var in ['SIZE', 'RISK', 'COST', 'LNBDP']:
    df_panel[f'd_{var}'] = df_panel[var] - df_panel.groupby(level='bank')[var].shift(1)

# Drop rows with missing values (due to lags/differencing)
df_diff = df_panel.dropna()
df_panel.head()

∟ Total assets (thousands USD)  \
bank Year                                    
DBS  2005                      108282660.0   
     2006                      128664923.0   
     2007                      161645150.0   
     2008                      178375490.0   
     2009                      184298135.0   

            ∟ Total equity / Total assets (%)  \
bank Year                                       
DBS  2005                               10.65   
     2006                               10.66   
     2007                                9.94   
     2008                                9.35   
     2009                               11.41   

            ∟ Total operating expenses (thousands USD)  \
bank Year                                                
DBS  2005                                    1217402.0   
     2006                                    1544328.0   
     2007                                    1816542.0   
     2008                                    1844775.0   
     2009                                    1855494.0   

            ∟ Net interest margin (average total earning assets) (%)  \
bank Year                                                              
DBS  2005                                               1.81           
     2006                                               2.13           
     2007                                               2.19           
     2008                                               2.00           
     2009                                               1.98           

            ∟ Net interest income (expenses) / Average total assets (%)  \
bank Year                                                                 
DBS  2005                                               1.63              
     2006                                               1.90              
     2007                                               1.91              
     2008                                               1.76              
     2009                                               1.73              

            ∟ Non-interest income / Average total assets (%)  \
bank Year                                                      
DBS  2005                                               0.77   
     2006                                               0.92   
     2007                                               0.95   
     2008                                               0.70   
     2009                                               0.83   

            ∟ Non-interest expenses / Average total assets (%)  \
bank Year                                                        
DBS  2005                                               1.26     
     2006                                               1.38     
     2007                                               1.28     
     2008                                               1.30     
     2009                                               1.56     

            ∟ Pre-tax operating profit / Average total assets (%)  \
bank Year                                                           
DBS  2005                                               0.74        
     2006                                               1.59        
     2007                                               1.41        
     2008                                               1.06        
     2009                                               0.98        

            ∟ Non-operating income (expense) & taxes / Average total assets (%)  \
bank Year                                                                         
DBS  2005                                              -0.60                      
     2006                                              -0.16                      
     2007                                              -0.45                      
     2008                                              -0.29                      
     2009    

# Run DIFF-GMM estimation using IV2SLS 

In [28]:
# Model specification (for ROA as an example):
# d_ROA = α * d_ROA_lag1 + β1*d_SIZE + β2*d_RISK + β3*d_COST + β4*d_LNBDP + β5*d_LNTPP + error
# Instrument: Use ROA_lag2 as an instrument for d_ROA_lag1

# Define dependent variable, exogenous regressors, the endogenous regressor, and instruments
dependent = df_diff['d_ROA']
exog = df_diff[['d_SIZE', 'd_RISK', 'd_COST', 'd_LNBDP']]
endog = df_diff['d_ROA_lag1']
instruments = df_diff[['ROA_lag2']]

# Fit the IV regression (this serves as our DIFF-GMM estimator for illustration)
mod = IV2SLS(dependent, exog, endog, instruments)
res = mod.fit(cov_type='robust')
print(res.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                  d_ROA   R-squared:                      0.6813
Estimator:                    IV-2SLS   Adj. R-squared:                 0.6089
No. Observations:                  27   F-statistic:                    92.731
Date:                Sun, Mar 30 2025   P-value (F-stat)                0.0000
Time:                        02:03:48   Distribution:                  chi2(5)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
d_SIZE        -0.6829     0.5606    -1.2181     0.2232     -1.7816      0.4159
d_RISK     -3.454e+07  1.537e+07    -2.2481     0.02

we only have BDP and not TPP (or calssify)
